# GSM8K power-policy baselines

This experiment supplies base sampling, development-selected low temperature, fixed-length RLOO and MCMC for the [adaptive-length study](../rloo_adapt_len/report.ipynb). The combined comparison table and uncertainty estimates appear there once.

## Protocol

Qwen2.5-0.5B, revision `060db6499f32faf8b98477b0a26969ef7d8b9987`, on the same 500 GSM8K test questions, 32 responses per question and a 512-token cap. The exact prompt is:

```text
Can you solve the following math problem? {question} Please reason step by step, and put your final answer within \boxed{{}}.
```

Low T=0.25 was chosen on separate development questions by mean answer pass@1/2/4/8. [Temperature selection](results/evaluation/temperature_selection.json) records the complete grid. The original [protocol](results/evaluation/protocol.json) is retained verbatim for provenance; it also records historical conditions outside this curated comparison.

MCMC uses target alpha 2, proposal T=0.5, block size 32 and two MH updates per block, with full-suffix acceptance and EOS retained. This is a finite-compute implementation, not evidence of convergence to the exact power distribution. All test conditions use the same model revision, prompt and question order.

## Fixed-length training and limitations

The historical alpha-2 run uses 1,024 GSM8K training questions, seed 0, all-linear LoRA rank 8 with LoRA scaling alpha 16, learning rate 1e-5 and a fixed 512-token cap. It completed 240 updates in 58.7 minutes. The objective and leave-one-out baseline are shared with adaptive RLOO; labels enter only monitoring. [Training config](results/fixed512_alpha2/config.json) and [metrics](results/fixed512_alpha2/metrics.jsonl) preserve the executed settings.

Fixed-length RLOO improves answer pass@1 over base T=1 but performs below low temperature and largely loses the requested boxed format. The adaptive comparison also changes update count and diagnostic overhead, so it does not isolate response-length scheduling alone.

## Reproduce the analysis

From the repository root, after installation:

```bash
python experiments/gsm8k_power_policy/scripts/analyze.py
python experiments/rloo_adapt_len/scripts/analyze.py
```

[Compact measurements](results/evaluation/summary.json) are recomputed from per-question correctness counts and token lengths. [Source hashes](results/provenance/manifest.json) link them to the original raw outputs, which remain local. Compact data supports pass@N, paired-question uncertainty and token cost analysis; it does not permit regrading response text. No model or GPU is needed to rebuild reports. See [SETUP](../../SETUP.md) for fresh-run entry points.